In [ ]:
def run_advanced_strategy_test():
    """运行高级策略测试"""
    print("🚀 开始高级策略回测比较")
    
    # 获取股票列表
    stock_codes = get_hs300_stock_list()[:20]  # 使用前20只股票进行快速测试
    print(f"使用 {len(stock_codes)} 只股票进行测试")
    
    # 获取数据
    stock_data_dict = generate_hs300_sample_data(
        stock_codes, 
        start_date='20250101', 
        end_date='20251231'
    )
    #print(stock_data_dict)
    if not stock_data_dict:
        print("错误: 无法获取股票数据")
        return
    
    # TODO:股票回测

    # 创建组合回测实例
    portfolio_backtest = PortfolioBacktest(initial_capital=1000000)
    
    # 运行组合回测
    print("开始组合回测...")
    portfolio_backtest.run_stock_universe_backtest(
        stock_data_dict=stock_data_dict,
        strategy_function=lambda x: AdvancedStrategies.volume_price_confirmation(x, 20),
        start_date='20250101',
        end_date='20251231',
        capital_per_stock=20000
    )
    
    # 生成报告
    #portfolio_backtest.print_detailed_report()
    
    # 绘制结果
    benchmark_codes = ['000300.SH']
    benchmark_data_dict = generate_hs300_sample_data(
        benchmark_codes, 
        start_date='20250101', 
        end_date='20251231'
    )
    #portfolio_backtest.plot_portfolio_performance(benchmark_data_dict)

    # 交易详情
    # portfolio_backtest.print_detailed_trade()

    # 每日持仓
    portfolio_backtest.print_detailed_position()
    return
    # 定义策略集合
    strategies = {
        "双均线交叉": lambda x: AdvancedStrategies.dual_moving_average_cross(x, 5, 20),
        "RSI均值回归": lambda x: AdvancedStrategies.rsi_mean_reversion(x, 14, 30, 70),
        "布林带突破": lambda x: AdvancedStrategies.bollinger_breakout(x, 20, 2),
        "MACD交叉": lambda x: AdvancedStrategies.macd_crossover(x, 12, 26, 9),
        "量价确认": lambda x: AdvancedStrategies.volume_price_confirmation(x, 20),
        "均值回归": lambda x: AdvancedStrategies.mean_reversion(x, 20, 2),
        "动量策略": lambda x: AdvancedStrategies.momentum_strategy(x, 10, 20)
    }
    
    # 运行多策略比较
    multi_backtest = MultiStrategyBacktest(initial_capital=500000)
    multi_backtest.run_strategy_comparison(
        stock_data_dict=stock_data_dict,
        strategies_dict=strategies,
        start_date='20250101',
        end_date='20251231',
        capital_per_stock=20000
    )
    
    # 生成比较报告
    #comparison_df = multi_backtest.print_strategy_comparison()
    
    # 绘制比较图表
    # multi_backtest.plot_strategy_comparison()
    
    # 显示最佳策略的详细报告
    #best_strategy_name = comparison_df['portfolio_total_return'].idxmax()
    #print(f"\n📊 最佳策略 '{best_strategy_name}' 的详细报告:")
    #print("=" * 70)
    #multi_backtest.strategy_results[best_strategy_name].print_detailed_report()

    return multi_backtest

# 主函数更新
def main():
    """主函数"""
    # 运行高级策略测试
    run_advanced_strategy_test()

if __name__ == "__main__":
    main()



In [ ]:
def generate_hs300_sample_data(stock_codes, start_date='20200101', end_date='20231231'):
    """生成沪深300股票的模拟数据"""
    date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    stock_data_dict = {}
    fields=['open', 'close', 'high', 'low', 'volume', 'amount', 'preClose']
    xtdata.download_history_data2(stock_codes, '1d', start_date, end_date)
    for i, code in enumerate(stock_codes):
        print(f"\n正在测试第 {i+1} 只股票: {code}")
        data = xtdata.get_market_data_ex(field_list=fields, stock_list=[code], 
                                         start_time=start_date, end_time=end_date, period='1d', 
                                         count=1000)
        if data and code in data:
            df = data[code]
            stock_data_dict[code] = df
            print(f"成功! 数据形状: {df.shape}, 列名: {df.columns.tolist()}")
        else:
            print("获取失败或数据格式异常") 
    return stock_data_dict
def get_hs300_stock_list():
    """
    获取沪深300成分股列表

    Returns:
        list: 包含沪深300成分股代码的列表，如果获取失败则返回空列表
    """
    try:
        # 获取沪深300成分股列表
        hs300_constituents = xtdata.get_stock_list_in_sector('沪深300')
        print(f"成功获取 {len(hs300_constituents)} 只沪深300成分股")
        print("前5只成分股示例:", hs300_constituents[:5])
        return hs300_constituents
    except Exception as e:
        print(f"获取沪深300成分股列表出错: {e}")
        return []  # 返回空列表而不是None，避免后续处理出错

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

import datetime
import dateutil
import os
import requests
from datetime import datetime, timedelta
import time
from dateutil import parser
import re

class StockBacktest:
    def __init__(self, initial_capital=100000, commission=0.001, slippage=0.001):
        self.initial_capital = initial_capital
        self.capital = initial_capital
        self.positions = 0
        self.trades = []
        self.portfolio_values = []
        self.dates = []
        self.commission = commission
        self.slippage = slippage
        self.position_price = 0
        self.current_stock = None

    def is_date_string_advanced(self, date_str):
        """
        使用dateutil库判断（更智能，能解析更多格式）
        """
        try:
            # 先做一些基本检查，避免解析像"12345"这样的纯数字字符串
            if re.match(r'^\d+$', date_str):
                # 纯数字字符串，只接受特定长度的
                if len(date_str) not in (4, 6, 8):
                    return False
            
            parser.parse(date_str, fuzzy=False)
            return True
        except (ValueError, TypeError, OverflowError):
            return False
        
    def run_backtest(self, data, strategy_function, stock_code=None, enable_stop=True):
        """
        运行单只股票的回测
        data: 包含价格数据的DataFrame，必须有'close'列
        strategy_function: 策略函数
        stock_code: 股票代码，用于记录
        """
        self.current_stock = stock_code
        self.data = data.copy()
        self.capital = self.initial_capital
        self.positions = 0
        self.trades = []
        self.portfolio_values = []
        self.dates = []
        
        for i in range(1, len(data)):
            current_data = data.iloc[:i]
            current_price = data.iloc[i]['close']
            
            # 获取日期
            if 'date' in data.columns:
                current_date = data.iloc[i]['date']
            elif self.is_date_string_advanced(data.index[i]):
                current_date = data.index[i]
            else:
                current_date = i
                
            # 获取交易信号
            signal = strategy_function(current_data)

            trade_subtype = None

            # 判断止损信号
            stop_loss = current_price < self.position_price * 0.95
            if stop_loss & enable_stop:
                signal = -1
                trade_subtype = 'STOP_LOSS'
            
            # 执行交易逻辑
            self.execute_trade(signal, trade_subtype, current_price, current_date)
            
            # 记录组合价值
            portfolio_value = self.capital + self.positions * current_price
            self.portfolio_values.append(portfolio_value)
            self.dates.append(current_date)
            
        return self.calculate_metrics()
    
    def execute_trade(self, signal, trade_subtype, price, date):
        """执行交易，考虑交易成本"""
        if signal == 1 and self.positions == 0:  # 买入信号，空仓
            # 考虑滑点和佣金
            execution_price = price * (1 + self.slippage)
            max_shares = self.capital // (execution_price * (1 + self.commission))
            
            if max_shares > 0:
                self.positions = max_shares
                cost = self.positions * execution_price * (1 + self.commission)
                self.capital -= cost
                self.position_price = execution_price
                self.trades.append({
                    'type': 'BUY', 
                    'date': date, 
                    'price': execution_price, 
                    'shares': self.positions,
                    'cost': cost,
                    'stock': self.current_stock
                })
            
        elif signal == -1 and self.positions > 0:  # 卖出信号，持仓
            execution_price = price * (1 - self.slippage)
            revenue = self.positions * execution_price * (1 - self.commission)
            self.capital += revenue
            
            # 计算这次交易的盈亏
            profit = revenue - (self.positions * self.position_price)
            
            self.trades.append({
                'type': 'SELL', 
                'sub_type': trade_subtype,
                'date': date, 
                'price': execution_price, 
                'shares': self.positions,
                'revenue': revenue,
                'profit': profit,
                'stock': self.current_stock
            })
            self.positions = 0
            self.position_price = 0
    
    def calculate_metrics(self):
        """计算回测指标"""
        if len(self.portfolio_values) == 0:
            return {}
            
        returns = pd.Series(self.portfolio_values).pct_change().dropna()
        
        if len(returns) == 0:
            return {}
        
        total_return = (self.portfolio_values[-1] - self.initial_capital) / self.initial_capital
        trading_days = len(self.portfolio_values)
        
        # 年化收益率（考虑实际交易天数）
        annual_return = (1 + total_return) ** (252 / trading_days) - 1 if trading_days > 0 else 0
        
        # 夏普比率
        if returns.std() > 0:
            sharpe_ratio = returns.mean() / returns.std() * np.sqrt(252)
        else:
            sharpe_ratio = 0
            
        max_drawdown = self.calculate_max_drawdown()
        
        # 胜率计算
        winning_trades = len([t for t in self.trades if t.get('profit', 0) > 0])
        total_trades = len([t for t in self.trades if 'profit' in t])
        win_rate = winning_trades / total_trades if total_trades > 0 else 0
        
        return {
            'stock_code': self.current_stock,
            'initial_capital': self.initial_capital,
            'final_value': self.portfolio_values[-1] if self.portfolio_values else self.initial_capital,
            'total_return': total_return,
            'annual_return': annual_return,
            'sharpe_ratio': sharpe_ratio,
            'max_drawdown': max_drawdown,
            'total_trades': len(self.trades),
            'win_rate': win_rate,
            'avg_trade_profit': np.mean([t.get('profit', 0) for t in self.trades if 'profit' in t]) if total_trades > 0 else 0
        }
    
    def calculate_max_drawdown(self):
        """计算最大回撤"""
        if not self.portfolio_values:
            return 0
            
        peak = self.portfolio_values[0]
        max_dd = 0
        
        for value in self.portfolio_values:
            if value > peak:
                peak = value
            dd = (peak - value) / peak
            if dd > max_dd:
                max_dd = dd
                
        return max_dd




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

from xtquant import xtdata
import datetime
import dateutil

import pandas as pd
import numpy as np
import os
import requests
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time

class PortfolioBacktest:
    def __init__(self, initial_capital=1000000):
        self.initial_capital = initial_capital
        self.stock_results = {}
        self.all_trades = []
        self.portfolio_values = {}
        
    def run_stock_universe_backtest(self, stock_data_dict, strategy_function, 
                                  start_date=None, end_date=None, 
                                  capital_per_stock=100000):
        """
        遍历股票池进行回测
        
        Parameters:
        stock_data_dict: 字典，键为股票代码，值为包含股票数据的DataFrame
        strategy_function: 策略函数
        start_date: 开始日期
        end_date: 结束日期
        capital_per_stock: 每只股票分配的资金
        """
        print(f"开始回测，股票数量: {len(stock_data_dict)}")
        print(f"时间范围: {start_date} 到 {end_date}")
        print(f"每只股票资金: {capital_per_stock:,}")
        print("=" * 60)
        
        total_stocks = len(stock_data_dict)
        completed = 0
        
        for stock_code, data in stock_data_dict.items():
            # 过滤时间范围
            if start_date and end_date:
                if 'date' in data.columns:
                    mask = (data['date'] >= start_date) & (data['date'] <= end_date)
                    filtered_data = data[mask].copy()
                else:
                    mask = (data.index >= start_date) & (data.index <= end_date)
                    filtered_data = data[mask].copy()
            else:
                filtered_data = data.copy()
            
            if len(filtered_data) < 50:  # 确保有足够的数据
                continue
                
            # 运行单只股票回测
            backtest = StockBacktest(initial_capital=capital_per_stock)
            metrics = backtest.run_backtest(filtered_data, strategy_function, stock_code)
            
            # 保存结果
            self.stock_results[stock_code] = metrics
            self.all_trades.extend(backtest.trades)
            
            # 保存组合价值序列（归一化以便比较）
            if backtest.portfolio_values:
                initial_value = backtest.portfolio_values[0] if backtest.portfolio_values else 1
                normalized_values = [v / initial_value for v in backtest.portfolio_values]
                self.portfolio_values[stock_code] = {
                    'dates': backtest.dates,
                    'values': normalized_values
                }
            
            completed += 1
            if completed % 10 == 0:
                print(f"进度: {completed}/{total_stocks}")
    
    def get_portfolio_metrics(self):
        """计算组合级别的回测指标"""
        if not self.stock_results:
            return {}
            
        # 组合总收益
        total_final_value = sum(result['final_value'] for result in self.stock_results.values())
        total_initial_value = sum(result['initial_capital'] for result in self.stock_results.values())
        portfolio_return = (total_final_value - total_initial_value) / total_initial_value

        # 平均指标
        avg_annual_return = np.mean([r['annual_return'] for r in self.stock_results.values()])
        avg_sharpe = np.mean([r['sharpe_ratio'] for r in self.stock_results.values()])
        avg_max_dd = np.mean([r['max_drawdown'] for r in self.stock_results.values()])
        avg_win_rate = np.mean([r['win_rate'] for r in self.stock_results.values()])
        
        # 正收益股票比例
        positive_returns = len([r for r in self.stock_results.values() if r['total_return'] > 0])
        positive_ratio = positive_returns / len(self.stock_results)

        return {
            'portfolio_total_return': portfolio_return,
            'avg_annual_return': avg_annual_return,
            'avg_sharpe_ratio': avg_sharpe,
            'avg_max_drawdown': avg_max_dd,
            'avg_win_rate': avg_win_rate,
            'positive_return_ratio': positive_ratio,
            'total_stocks_tested': len(self.stock_results),
            'total_trades': len(self.all_trades),
            'total_final_value': total_final_value
        }
    
    def print_detailed_report(self):
        """打印详细的组合回测报告"""
        portfolio_metrics = self.get_portfolio_metrics()
        
        print("=" * 70)
        print("PORTFOLIO BACKTEST REPORT - 沪深300股票池")
        print("=" * 70)
        print(f"测试股票数量: {portfolio_metrics['total_stocks_tested']}")
        print(f"总交易次数: {portfolio_metrics['total_trades']}")
        print(f"组合总收益率: {portfolio_metrics['portfolio_total_return']:.2%}")
        print(f"平均年化收益率: {portfolio_metrics['avg_annual_return']:.2%}")
        print(f"平均夏普比率: {portfolio_metrics['avg_sharpe_ratio']:.2f}")
        print(f"平均最大回撤: {portfolio_metrics['avg_max_drawdown']:.2%}")
        print(f"平均胜率: {portfolio_metrics['avg_win_rate']:.2%}")
        print(f"正收益股票比例: {portfolio_metrics['positive_return_ratio']:.2%}")
        print(f"最终组合价值: ${portfolio_metrics['total_final_value']:,.2f}")
        print("=" * 70)
        
        # 显示表现最好和最差的股票
        if self.stock_results:
            sorted_stocks = sorted(self.stock_results.items(), 
                                 key=lambda x: x[1]['total_return'], 
                                 reverse=True)
            
            print("\n表现最好的5只股票:")
            for stock, metrics in sorted_stocks[:5]:
                print(f"  {stock}: {metrics['total_return']:.2%} (交易次数: {metrics['total_trades']})")
            
            print("\n表现最差的5只股票:")
            for stock, metrics in sorted_stocks[-5:]:
                print(f"  {stock}: {metrics['total_return']:.2%} (交易次数: {metrics['total_trades']})")
    
    def plot_portfolio_performance(self, benchmark_data_dict):
        """绘制组合表现"""
        if not self.portfolio_values:
            print("没有足够的数据进行绘图")
            return
            
        plt.figure(figsize=(15, 12))
        
        # 1. 所有股票的归一化收益曲线
        plt.subplot(2, 2, 1)
        for stock_code, data in list(self.portfolio_values.items())[:20]:  # 只显示前20只股票
            if len(data['dates']) > 0:
                plt.plot(data['dates'], data['values'], alpha=0.3, linewidth=1)
        
        # 计算平均曲线
        all_dates = set()
        for data in self.portfolio_values.values():
            all_dates.update(data['dates'])
        
        if all_dates:
            sorted_dates = sorted(all_dates)
            avg_values = []
            for date in sorted_dates:
                day_values = []
                for data in self.portfolio_values.values():
                    if date in data['dates']:
                        idx = data['dates'].index(date)
                        day_values.append(data['values'][idx])
                if day_values:
                    avg_values.append(np.mean(day_values))
            
            if len(avg_values) == len(sorted_dates):
                plt.plot(sorted_dates, avg_values, 'b-', linewidth=3, label='AvgReturn')

        if benchmark_data_dict:
            for benchmark_code, data in benchmark_data_dict.items():
                data['norm_value'] = data['close'] / data['close'][0]
                plt.plot(data['norm_value'], 'r--', linewidth=2, label='Benchmark')
                plt.xticks(range(1, len(data.index), 5), rotation=75)
              
        plt.title('Norm Curve')
        plt.ylabel('Norm')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 2. 收益率分布
        plt.subplot(2, 2, 2)
        returns = [metrics['total_return'] for metrics in self.stock_results.values()]
        plt.hist(returns, bins=30, alpha=0.7, edgecolor='black')
        plt.axvline(x=np.mean(returns), color='red', linestyle='--', label=f'AvgReturn: {np.mean(returns):.2%}')
        plt.title('Stock Return Dist')
        plt.xlabel('Return')
        plt.ylabel('Num of Stocks')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 3. 夏普比率分布
        plt.subplot(2, 2, 3)
        sharpes = [metrics['sharpe_ratio'] for metrics in self.stock_results.values()]
        plt.hist(sharpes, bins=30, alpha=0.7, edgecolor='black', color='green')
        plt.axvline(x=np.mean(sharpes), color='red', linestyle='--', label=f'AvgSharpe: {np.mean(sharpes):.2f}')
        plt.title('Sharpe Ratio Dist')
        plt.xlabel('Sharpe Ratio')
        plt.ylabel('Num of Stocks')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 4. 最大回撤分布
        plt.subplot(2, 2, 4)
        drawdowns = [metrics['max_drawdown'] for metrics in self.stock_results.values()]
        plt.hist(drawdowns, bins=30, alpha=0.7, edgecolor='black', color='orange')
        plt.axvline(x=np.mean(drawdowns), color='red', linestyle='--', label=f'avg: {np.mean(drawdowns):.2%}')
        plt.title('Max Drawdown Dist')
        plt.xlabel('Max Drawdown')
        plt.ylabel('Num of Stocks')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

    def print_detailed_trade(self):
        """打印详细的交易报告"""
        print("=" * 70)
        print("PORTFOLIO BACKTEST REPORT - 交易详情")
        print("=" * 70)
        df = pd.DataFrame(self.all_trades)
        trades_df = df.reindex(columns=['date', 'stock', 'type', 'sub_type', 'shares', 'price', 'cost', 'revenue', 'profit'])
        
        print(trades_df.sort_values(['date']))

    def print_detailed_position(self):
        """打印详细的每日持仓"""
        print("=" * 70)
        print("PORTFOLIO BACKTEST REPORT - 每日持仓")
        print("=" * 70)

        print(self.portfolio_values)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

from xtquant import xtdata
import datetime
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

class AdvancedStrategies:
    """高级策略库"""
    
    @staticmethod
    def dual_moving_average_cross(data, short_window=5, long_window=20):
        """双均线交叉策略"""
        if len(data) < long_window:
            return 0
        
        df = data.copy()
        df['short_ma'] = df['close'].rolling(window=short_window).mean()
        df['long_ma'] = df['close'].rolling(window=long_window).mean()
        
        current_short = df['short_ma'].iloc[-1]
        current_long = df['long_ma'].iloc[-1]
        
        if len(df) > 1:
            prev_short = df['short_ma'].iloc[-2]
            prev_long = df['long_ma'].iloc[-2]
        else:
            return 0
        
        # 金叉买入，死叉卖出
        if prev_short <= prev_long and current_short > current_long:
            return 1
        elif prev_short >= prev_long and current_short < current_long:
            return -1
        else:
            return 0
    
    @staticmethod
    def rsi_mean_reversion(data, period=14, oversold=30, overbought=70):
        """RSI均值回归策略"""
        if len(data) < period + 1:
            return 0
        
        df = data.copy()
        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        
        current_rsi = rsi.iloc[-1]
        
        if current_rsi < oversold:
            return 1  # 超卖，买入
        elif current_rsi > overbought:
            return -1  # 超买，卖出
        else:
            return 0
    
    @staticmethod
    def bollinger_breakout(data, period=20, num_std=2):
        """布林带突破策略"""
        if len(data) < period:
            return 0
        
        df = data.copy()
        df['middle_band'] = df['close'].rolling(window=period).mean()
        df['std'] = df['close'].rolling(window=period).std()
        df['upper_band'] = df['middle_band'] + (df['std'] * num_std)
        df['lower_band'] = df['middle_band'] - (df['std'] * num_std)
        
        current_close = df['close'].iloc[-1]
        current_upper = df['upper_band'].iloc[-1]
        current_lower = df['lower_band'].iloc[-1]
        prev_close = df['close'].iloc[-2] if len(df) > 1 else current_close
        
        # 上突破买入，下突破卖出
        if prev_close <= current_upper and current_close > current_upper:
            return 1
        elif prev_close >= current_lower and current_close < current_lower:
            return -1
        else:
            return 0
    
    @staticmethod
    def macd_crossover(data, fast_period=12, slow_period=26, signal_period=9):
        """MACD交叉策略"""
        if len(data) < slow_period + signal_period:
            return 0
        
        df = data.copy()
        exp1 = df['close'].ewm(span=fast_period, adjust=False).mean()
        exp2 = df['close'].ewm(span=slow_period, adjust=False).mean()
        macd = exp1 - exp2
        signal = macd.ewm(span=signal_period, adjust=False).mean()
        histogram = macd - signal
        
        current_macd = macd.iloc[-1]
        current_signal = signal.iloc[-1]
        prev_macd = macd.iloc[-2] if len(macd) > 1 else current_macd
        prev_signal = signal.iloc[-2] if len(signal) > 1 else current_signal
        
        # MACD上穿信号线买入，下穿信号线卖出
        if prev_macd <= prev_signal and current_macd > current_signal:
            return 1
        elif prev_macd >= prev_signal and current_macd < current_signal:
            return -1
        else:
            return 0
    
    @staticmethod
    def volume_price_confirmation(data, volume_period=20):
        """量价确认策略"""
        if len(data) < volume_period:
            return 0
        
        df = data.copy()
        df['price_change'] = df['close'].pct_change()
        df['volume_ma'] = df['volume'].rolling(window=volume_period).mean()
        df['volume_ratio'] = df['volume'] / df['volume_ma']
        
        current_price_change = df['price_change'].iloc[-1]
        current_volume_ratio = df['volume_ratio'].iloc[-2]
        prev_price_change = df['price_change'].iloc[-2] if len(df) > 1 else 0
        
        # 买入信号
        # 价涨量增买入，价跌量跌买入
        if (current_price_change > 0 and current_volume_ratio > 3) or \
           (current_price_change < 0 and current_volume_ratio < 0.5):
            return 1
        
        # 卖出信号  
        # 价涨量跌卖出，价跌量增卖出
        elif (current_price_change < 0 and current_volume_ratio > 3) or \
             (current_price_change > 0 and current_volume_ratio < 0.5):
            return -1
        
        else:
            return 0

    @staticmethod
    def mean_reversion(data, lookback=20, z_threshold=2):
        """均值回归策略（Z-score）"""
        if len(data) < lookback:
            return 0
        
        df = data.copy()
        returns = df['close'].pct_change().dropna()
        
        if len(returns) < lookback:
            return 0
        
        current_return = returns.iloc[-1]
        mean_return = returns.tail(lookback).mean()
        std_return = returns.tail(lookback).std()
        
        if std_return == 0:
            return 0
        
        z_score = (current_return - mean_return) / std_return
        
        # Z-score极端值回归
        if z_score < -z_threshold:
            return 1  # 超卖回归
        elif z_score > z_threshold:
            return -1  # 超买回归
        else:
            return 0
    
    @staticmethod
    def momentum_strategy(data, momentum_period=10, ma_period=20):
        """动量策略"""
        if len(data) < max(momentum_period, ma_period):
            return 0
        
        df = data.copy()
        df['momentum'] = df['close'] / df['close'].shift(momentum_period) - 1
        df['ma'] = df['close'].rolling(window=ma_period).mean()
        
        current_momentum = df['momentum'].iloc[-1]
        current_close = df['close'].iloc[-1]
        current_ma = df['ma'].iloc[-1]
        
        # 动量强劲且价格在均线上方买入
        if current_momentum > 0.02 and current_close > current_ma:
            return 1
        # 负动量且价格在均线下方卖出
        elif current_momentum < -0.02 and current_close < current_ma:
            return -1
        else:
            return 0



In [ ]:
class MultiStrategyBacktest:
    """多策略回测框架"""
    
    def __init__(self, initial_capital=1000000):
        self.initial_capital = initial_capital
        self.strategy_results = {}
        self.comparison_results = {}
        
    def run_strategy_comparison(self, stock_data_dict, strategies_dict, 
                              start_date=None, end_date=None,
                              capital_per_stock=100000):
        """
        运行多策略比较回测
        
        Parameters:
        stock_data_dict: 股票数据字典
        strategies_dict: 策略字典，{策略名称: 策略函数}
        """
        print("=" * 80)
        print("多策略回测比较")
        print("=" * 80)
        print(f"策略数量: {len(strategies_dict)}")
        print(f"股票数量: {len(stock_data_dict)}")
        print(f"时间范围: {start_date} 到 {end_date}")
        print("=" * 80)
        
        for strategy_name, strategy_func in strategies_dict.items():
            print(f"\n正在运行策略: {strategy_name}")
            
            portfolio_backtest = PortfolioBacktest(initial_capital=self.initial_capital)
            portfolio_backtest.run_stock_universe_backtest(
                stock_data_dict=stock_data_dict,
                strategy_function=strategy_func,
                start_date=start_date,
                end_date=end_date,
                capital_per_stock=capital_per_stock
            )
            
            # 保存策略结果
            self.strategy_results[strategy_name] = portfolio_backtest
            self.comparison_results[strategy_name] = portfolio_backtest.get_portfolio_metrics()
            
            # 打印策略简要结果
            metrics = self.comparison_results[strategy_name]
            print(f"{strategy_name} - 总收益: {metrics['portfolio_total_return']:.2%} | "
                  f"年化收益: {metrics['avg_annual_return']:.2%} | "
                  f"夏普比率: {metrics['avg_sharpe_ratio']:.2f}")
    
    def print_strategy_comparison(self):
        """打印策略比较报告"""
        if not self.comparison_results:
            print("没有可比较的结果")
            return
        
        print("\n" + "=" * 100)
        print("多策略比较报告")
        print("=" * 100)
        
        # 创建比较表格
        comparison_df = pd.DataFrame(self.comparison_results).T
        comparison_df = comparison_df.sort_values('portfolio_total_return', ascending=False)
        
        # 选择关键指标显示
        key_metrics = [
            'portfolio_total_return', 'avg_annual_return', 'avg_sharpe_ratio',
            'avg_max_drawdown', 'avg_win_rate', 'positive_return_ratio', 'total_trades'
        ]
        
        display_df = comparison_df[key_metrics].copy()
        display_df.columns = ['总收益率', '年化收益率', '夏普比率', '平均最大回撤', 
                            '平均胜率', '正收益比例', '总交易次数']
        
        # 格式化显示
        formatted_df = display_df.copy()
        formatted_df['总收益率'] = formatted_df['总收益率'].apply(lambda x: f"{x:.2%}")
        formatted_df['年化收益率'] = formatted_df['年化收益率'].apply(lambda x: f"{x:.2%}")
        formatted_df['平均最大回撤'] = formatted_df['平均最大回撤'].apply(lambda x: f"{x:.2%}")
        formatted_df['平均胜率'] = formatted_df['平均胜率'].apply(lambda x: f"{x:.2%}")
        formatted_df['正收益比例'] = formatted_df['正收益比例'].apply(lambda x: f"{x:.2%}")
        formatted_df['夏普比率'] = formatted_df['夏普比率'].apply(lambda x: f"{x:.2f}")
        
        print(formatted_df.to_string())
        print("=" * 100)
        
        # 找出最佳策略
        best_strategy = comparison_df['portfolio_total_return'].idxmax()
        best_return = comparison_df.loc[best_strategy, 'portfolio_total_return']
        
        print(f"\n🎯 最佳策略: {best_strategy} (总收益: {best_return:.2%})")
        
        return comparison_df
    
    def plot_strategy_comparison(self):
        """绘制策略比较图"""
        if not self.strategy_results:
            print("没有策略结果可比较")
            return
        
        plt.figure(figsize=(16, 12))
        
        # 1. 策略收益对比
        plt.subplot(2, 2, 1)
        strategy_returns = {name: result.get_portfolio_metrics()['portfolio_total_return'] 
                          for name, result in self.strategy_results.items()}
        
        colors = plt.cm.Set3(np.linspace(0, 1, len(strategy_returns)))
        bars = plt.bar(strategy_returns.keys(), strategy_returns.values(), color=colors)
        plt.title('策略总收益对比')
        plt.ylabel('总收益率')
        plt.xticks(rotation=45)
        
        # 在柱状图上添加数值
        for bar, value in zip(bars, strategy_returns.values()):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{value:.2%}', ha='center', va='bottom')
        
        # 2. 夏普比率对比
        plt.subplot(2, 2, 2)
        strategy_sharpes = {name: result.get_portfolio_metrics()['avg_sharpe_ratio'] 
                          for name, result in self.strategy_results.items()}
        
        colors = plt.cm.Pastel1(np.linspace(0, 1, len(strategy_sharpes)))
        bars = plt.bar(strategy_sharpes.keys(), strategy_sharpes.values(), color=colors)
        plt.title('策略夏普比率对比')
        plt.ylabel('夏普比率')
        plt.xticks(rotation=45)
        
        # 3. 最大回撤对比
        plt.subplot(2, 2, 3)
        strategy_drawdowns = {name: result.get_portfolio_metrics()['avg_max_drawdown'] 
                            for name, result in self.strategy_results.items()}
        
        colors = plt.cm.Set2(np.linspace(0, 1, len(strategy_drawdowns)))
        bars = plt.bar(strategy_drawdowns.keys(), strategy_drawdowns.values(), color=colors)
        plt.title('策略最大回撤对比')
        plt.ylabel('最大回撤')
        plt.xticks(rotation=45)
        
        # 4. 胜率对比
        plt.subplot(2, 2, 4)
        strategy_winrates = {name: result.get_portfolio_metrics()['avg_win_rate'] 
                           for name, result in self.strategy_results.items()}
        
        colors = plt.cm.Paired(np.linspace(0, 1, len(strategy_winrates)))
        bars = plt.bar(strategy_winrates.keys(), strategy_winrates.values(), color=colors)
        plt.title('策略胜率对比')
        plt.ylabel('胜率')
        plt.xticks(rotation=45)
        
        plt.tight_layout()
        plt.show()




In [ ]:
test = [{'type': 'BUY', 'date': '20250409', 'price': 33.19316, 'shares': 601.0, 'cost': 19969.038249159996, 'stock': '600009.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250428', 'price': 32.14782, 'shares': 601.0, 'revenue': 19301.51898018, 'profit': -647.5701798199989, 'stock': '600009.SH'}, {'type': 'BUY', 'date': '20250429', 'price': 32.11208, 'shares': 601.0, 'cost': 19318.659440079995, 'stock': '600009.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250506', 'price': 32.21775, 'shares': 601.0, 'revenue': 19343.504882250003, 'profit': 44.14480225000443, 'stock': '600009.SH'}, {'type': 'BUY', 'date': '20250403', 'price': 1.8017999999999998, 'shares': 11088.0, 'cost': 19998.336758399993, 'stock': '600010.SH'}, {'type': 'SELL', 'sub_type': 'STOP_LOSS', 'date': '20250407', 'price': 1.6483500000000002, 'shares': 11088.0, 'revenue': 18258.627895200003, 'profit': -1719.7305047999944, 'stock': '600010.SH'}, {'type': 'BUY', 'date': '20250408', 'price': 1.71171, 'shares': 10657.0, 'cost': 18259.93516347, 'stock': '600010.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250422', 'price': 1.7782200000000001, 'shares': 10657.0, 'revenue': 18931.540049460004, 'profit': 689.8465794600015, 'stock': '600010.SH'}, {'type': 'BUY', 'date': '20250529', 'price': 1.75175, 'shares': 10796.0, 'cost': 18930.804892999997, 'stock': '600010.SH'}, {'type': 'BUY', 'date': '20250423', 'price': 7.087079999999999, 'shares': 2819.0, 'cost': 19998.456998519996, 'stock': '600011.SH'}, {'type': 'BUY', 'date': '20250318', 'price': 7.627619999999999, 'shares': 2619.0, 'cost': 19996.713516779997, 'stock': '600015.SH'}, {'type': 'SELL', 'sub_type': 'STOP_LOSS', 'date': '20250506', 'price': 7.1928, 'shares': 2619.0, 'revenue': 18819.1052568, 'profit': -1157.631523199998, 'stock': '600015.SH'}, {'type': 'BUY', 'date': '20250422', 'price': 5.52552, 'shares': 3615.0, 'cost': 19994.7295548, 'stock': '600018.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250429', 'price': 5.474519999999999, 'shares': 3615.0, 'revenue': 19770.599410199997, 'profit': -204.15538980000565, 'stock': '600018.SH'}, {'type': 'BUY', 'date': '20250331', 'price': 7.207199999999999, 'shares': 2772.0, 'cost': 19998.336758399993, 'stock': '600019.SH'}, {'type': 'SELL', 'sub_type': 'STOP_LOSS', 'date': '20250407', 'price': 6.6833100000000005, 'shares': 2772.0, 'revenue': 18507.60918468, 'profit': -1470.7492153199964, 'stock': '600019.SH'}, {'type': 'BUY', 'date': '20250519', 'price': 6.97697, 'shares': 2650.0, 'cost': 18507.459470499998, 'stock': '600019.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250522', 'price': 6.853140000000001, 'shares': 2650.0, 'revenue': 18142.660179000002, 'profit': -346.3103209999972, 'stock': '600019.SH'}, {'type': 'BUY', 'date': '20250422', 'price': 9.359350000000001, 'shares': 2134.0, 'cost': 19992.8257529, 'stock': '600025.SH'}, {'type': 'BUY', 'date': '20250407', 'price': 9.89989, 'shares': 2018.0, 'cost': 19997.955998019996, 'stock': '600026.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250529', 'price': 10.14984, 'shares': 2018.0, 'revenue': 20461.89474288, 'profit': 483.9167228799997, 'stock': '600026.SH'}, {'type': 'BUY', 'date': '20250605', 'price': 10.060049999999999, 'shares': 2032.0, 'cost': 20462.463621599993, 'stock': '600026.SH'}, {'type': 'BUY', 'date': '20250422', 'price': 5.69569, 'shares': 3507.0, 'cost': 19994.75961483, 'stock': '600028.SH'}, {'type': 'BUY', 'date': '20250423', 'price': 5.705699999999999, 'shares': 3501.0, 'cost': 19995.631355699996, 'stock': '600029.SH'}, {'type': 'BUY', 'date': '20250401', 'price': 18.94893, 'shares': 1054.0, 'cost': 19992.144392219998, 'stock': '600031.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250403', 'price': 19.0809, 'shares': 1054.0, 'revenue': 20091.157331399998, 'profit': 118.98511139999755, 'stock': '600031.SH'}, {'type': 'BUY', 'date': '20250530', 'price': 43.47342999999999, 'shares': 459.0, 'cost': 19974.258674369998, 'stock': '600036.SH'}, {'type': 'BUY', 'date': '20250331', 'price': 7.947939999999999, 'shares': 2513.0, 'cost': 19993.146393219995, 'stock': '600039.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250417', 'price': 8.5914, 'shares': 2513.0, 'revenue': 21568.5980118, 'profit': 1595.4247918000037, 'stock': '600039.SH'}, {'type': 'BUY', 'date': '20250422', 'price': 8.5085, 'shares': 2533.0, 'cost': 21573.5825305, 'stock': '600039.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250519', 'price': 9.1908, 'shares': 2533.0, 'revenue': 23257.0161036, 'profit': 1704.9856035999983, 'stock': '600039.SH'}, {'type': 'BUY', 'date': '20250526', 'price': 9.26926, 'shares': 2506.0, 'cost': 23251.994325559994, 'stock': '600039.SH'}, {'type': 'BUY', 'date': '20250522', 'price': 8.17817, 'shares': 2443.0, 'cost': 19999.24857931, 'stock': '600048.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250530', 'price': 8.121870000000001, 'shares': 2443.0, 'revenue': 19821.886681590004, 'profit': -157.38262840999596, 'stock': '600048.SH'}, {'type': 'BUY', 'date': '20250211', 'price': 5.485479999999999, 'shares': 3642.0, 'cost': 19998.096278159996, 'stock': '600050.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250219', 'price': 6.43356, 'shares': 3642.0, 'revenue': 23407.59449448, 'profit': 3429.4763344800012, 'stock': '600050.SH'}, {'type': 'BUY', 'date': '20250311', 'price': 6.216209999999999, 'shares': 3762.0, 'cost': 23408.767402019996, 'stock': '600050.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250313', 'price': 6.12387, 'shares': 3762.0, 'revenue': 23014.960941060002, 'profit': -370.4210789399949, 'stock': '600050.SH'}, {'type': 'BUY', 'date': '20250318', 'price': 6.216209999999999, 'shares': 3698.0, 'cost': 23010.532124579997, 'stock': '600050.SH'}, {'type': 'SELL', 'sub_type': 'STOP_LOSS', 'date': '20250319', 'price': 5.76423, 'shares': 3698.0, 'revenue': 21294.80641746, 'profit': -1692.7381625399976, 'stock': '600050.SH'}, {'type': 'BUY', 'date': '20250328', 'price': 5.57557, 'shares': 3816.0, 'cost': 21297.651495119997, 'stock': '600050.SH'}, {'type': 'SELL', 'sub_type': None, 'date': '20250403', 'price': 5.584410000000001, 'shares': 3816.0, 'revenue': 21288.798451440005, 'profit': 12.423331440004404, 'stock': '600050.SH'}, {'type': 'BUY', 'date': '20250506', 'price': 5.48548, 'shares': 3877.0, 'cost': 21288.473165959997, 'stock': '600050.SH'}]

In [ ]:
df = pd.DataFrame(test)
df.columns
df.head()

In [ ]:
df = df.reindex(columns=['date', 'stock', 'type', 'sub_type', 'shares', 'price', 'cost', 'revenue', 'profit'])
df.sort_values(['date'])

In [ ]:
df

In [ ]:
test = {'600000.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, '600009.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980060875420004, 0.9937990875420001, 0.982981087542, 0.982680587542, 0.9685570875420002, 0.984183087542, 0.991695587542, 0.9787740875420002, 0.9751680875420001, 0.9724635875420002, 0.9655520875420002, 0.965852587542, 0.9709610875420002, 0.9666240365510003, 0.9646950645470006, 0.9652960645470006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006, 0.9678663086595006]}, '600010.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980031620800004, 0.9130145568400004, 0.9111912986665005, 0.9325052986665004, 0.9378337986665005, 0.9484907986665004, 0.9431622986665005, 0.9325052986665004, 0.9378337986665005, 0.9431622986665005, 0.9378337986665005, 0.9538192986665004, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9465948011395007, 0.9447045564895008, 0.9393065564895009, 0.9339085564895008, 0.9501025564895008, 0.9447045564895008]}, '600011.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980031500740002, 1.0120981500740003, 1.012098150074, 1.0233741500740001, 1.0149171500740002, 1.0092791500740002, 1.0374691500740003, 1.0304216500740002, 1.0318311500740003, 1.0501546500740002, 1.0473356500740003, 1.0487451500740002, 1.0445166500740002, 1.0572021500740003, 1.0543831500740002, 1.0529736500740001, 1.0600211500740002, 1.0557926500740002, 1.0431071500740001, 1.0416976500740003, 1.0501546500740002, 1.0473356500740003, 1.0191456500740002, 1.0177361500740003, 1.0191456500740002, 1.0247836500740002, 1.0149171500740002, 1.0149171500740002]}, '600015.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980033241610001, 1.003241324161, 1.0019318241610002, 0.9849083241610002, 0.9953843241610001, 1.0045508241610004, 0.9993128241610002, 1.0019318241610002, 1.0137173241610002, 1.0255028241610002, 1.0255028241610002, 1.038597824161, 1.0385978241610003, 0.9705038241610002, 0.9888368241610002, 0.9875273241610002, 0.9914558241610001, 0.9914558241610002, 1.003241324161, 1.0255028241610002, 1.0425263241610003, 1.038597824161, 1.0438358241610002, 1.0320503241610002, 1.0425263241610003, 1.0294313241610002, 1.0477643241610002, 1.0451453241610003, 1.054311824161, 1.041216824161, 0.9521708241610002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002, 0.9411195870010002]}, '600016.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, '600018.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980035222600001, 0.9925810222600001, 0.9907735222599998, 0.99438852226, 1.0034260222600002, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998, 0.9887934927699998]}, '600019.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980031620800004, 1.0146351620800003, 1.0021611620800004, 1.0063191620800005, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9254636213140004, 0.9236156477890005, 0.9196406477890005, 0.9209656477890005, 0.9072236567390006, 0.9072236567390006, 0.9072236567390006, 0.9072236567390006, 0.9072236567390006, 0.9072236567390006, 0.9072236567390006, 0.9072236567390006, 0.9072236567390006, 0.9072236567390006]}, '600023.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, '600025.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980037123550001, 0.9851997123549999, 0.991601712355, 1.006539712355, 1.022544712355, 1.0172097123550001, 1.0193437123549998, 1.012941712355, 1.0225447123550002, 1.0268127123550002, 1.046018712355, 1.039616712355, 1.0364157123550002, 1.035348712355, 1.041750712355, 1.037482712355, 1.054554712355, 1.066291712355, 1.072693712355, 1.0790957123549998, 1.066291712355, 1.070559712355, 1.075894712355, 1.0716267123549998, 1.065224712355, 1.074827712355, 1.058822712355, 1.042817712355, 1.035348712355]}, '600026.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980032000990002, 1.0080932000990002, 1.017174200099, 1.0343272000990003, 1.0262552000990002, 1.0302912000990003, 1.0141472000990004, 1.0302912000990003, 1.0181832000990003, 1.0101112000990002, 1.0060752000990003, 1.0111202000990005, 1.0151562000990002, 1.028273200099, 1.0403812000990003, 1.0666152000990003, 1.053498200099, 1.0383632000990002, 1.0827592000990003, 1.077714200099, 1.0595522000990003, 1.0514802000990002, 1.062579200099, 1.0676242000990002, 1.1120202000990003, 1.0827592000990003, 1.064597200099, 1.0656062000990003, 1.0504712000990002, 1.0464352000990005, 1.0313002000990001, 1.0191922000990001, 1.029282200099, 1.022219200099, 1.0222192000990002, 1.023196937243, 1.023196937243, 1.023196937243, 1.023196937243, 1.0211537561630004]}, '600027.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, '600028.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980035192585001, 0.9962500192584999, 0.9997570192585, 0.9962500192584999, 1.0015105192585, 0.9874825192585, 0.9927430192585, 0.9892360192585, 1.0015105192585, 0.9980035192585, 0.9980035192585, 0.9997570192584999, 1.0050175192584998, 1.0225525192585, 1.0137850192585, 0.9927430192585, 0.9962500192584999, 0.9944965192584999, 1.0015105192585, 1.0137850192585, 0.9980035192585001, 0.9962500192584999, 0.9962500192584999, 1.0155385192585, 1.0050175192585, 1.0137850192585, 1.0085245192585, 1.0102780192585, 1.0032640192585]}, '600029.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980034322150002, 0.9980034322150002, 0.9910014322150003, 0.9875004322150002, 0.9962529322150003, 0.9839994322150002, 1.0347639322150002, 1.015508432215, 1.0137579322150003, 1.0067559322150001, 1.0330134322150002, 1.0382649322150002, 1.0382649322150004, 1.0330134322150002, 1.045266932215, 1.0400154322150001, 1.0452669322150003, 1.0645224322150002, 1.0697739322150002, 1.052268932215, 1.089029432215, 1.0855284322150003, 1.0925304322150002, 1.0732749322150004, 1.0662729322150002, 1.0575204322150003, 1.0400154322150001, 1.0435164322150001]}, '600030.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}, '600031.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980037803890004, 1.020664780389, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959, 1.004950646959]}, '600036.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980055662815001, 1.0103985662815, 1.0124640662814999, 1.0195785662815]}, '600039.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980036803390003, 1.0055426803390002, 1.0432376803390002, 1.0721371803390003, 1.0495201803390002, 1.0759066803390003, 1.0934976803390004, 1.087215180339, 1.0633416803390003, 1.0683676803390003, 1.0558026803390002, 1.0796761803390005, 1.0787725809290003, 1.0787725809290003, 1.0787725809290003, 1.0766184544040003, 1.0766184544040005, 1.1209459544040004, 1.1222124544040006, 1.1184129544040002, 1.1285449544040003, 1.1006819544040003, 1.1285449544040003, 1.1450094544040004, 1.1462759544040004, 1.1754054544040002, 1.1690729544040004, 1.1741389544040002, 1.1741389544040002, 1.176671954404, 1.1792049544040002, 1.1629442595840003, 1.1629442595840003, 1.1629442595840003, 1.1629442595840003, 1.1629442595840003, 1.1606225433060005, 1.1606225433060005, 1.1693935433060005, 1.160622543306001, 1.160622543306001, 1.1656345433060007, 1.1693935433060005, 1.1806705433060007]}, '600048.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980030710345001, 0.9882310710345, 0.9857880710345002, 0.9894525710345, 0.9870095710345, 0.9906740710345002, 0.9911319051140003, 0.9911319051140003, 0.9911319051140003, 0.9911319051140003]}, '600050.SH': {'dates': ['20250103', '20250106', '20250107', '20250108', '20250109', '20250110', '20250113', '20250114', '20250115', '20250116', '20250117', '20250120', '20250121', '20250122', '20250123', '20250124', '20250127', '20250205', '20250206', '20250207', '20250210', '20250211', '20250212', '20250213', '20250214', '20250217', '20250218', '20250219', '20250220', '20250221', '20250224', '20250225', '20250226', '20250227', '20250228', '20250303', '20250304', '20250305', '20250306', '20250307', '20250310', '20250311', '20250312', '20250313', '20250314', '20250317', '20250318', '20250319', '20250320', '20250321', '20250324', '20250325', '20250326', '20250327', '20250328', '20250331', '20250401', '20250402', '20250403', '20250407', '20250408', '20250409', '20250410', '20250411', '20250414', '20250415', '20250416', '20250417', '20250418', '20250421', '20250422', '20250423', '20250424', '20250425', '20250428', '20250429', '20250430', '20250506', '20250507', '20250508', '20250509', '20250512', '20250513', '20250514', '20250515', '20250516', '20250519', '20250520', '20250521', '20250522', '20250523', '20250526', '20250527', '20250528', '20250529', '20250530', '20250603', '20250604', '20250605'], 'values': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9980031860920002, 1.0016451860920002, 1.0016451860920002, 1.0890531860920003, 1.1855661860920002, 1.1691771860920002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1704749108160002, 1.1681375407150005, 1.1718995407150006, 1.1507845877680005, 1.1507845877680005, 1.1507845877680005, 1.1484869815390006, 1.0649983024120007, 1.0649983024120007, 1.0649983024120007, 1.0649983024120007, 1.0649983024120007, 1.0649983024120007, 1.0649983024120007, 1.062871727656001, 1.0609637276560009, 1.0609637276560009, 1.0647797276560007, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0645556502280011, 1.0624299919300013, 1.0624299919300013, 1.0585529919300012, 1.0527374919300012, 1.0624299919300013, 1.054675991930001, 1.0740609919300013, 1.054675991930001, 1.0430449919300013, 1.0507989919300014, 1.0604914919300013, 1.0585529919300012, 1.0527374919300012, 1.043044991930001, 1.0469219919300015, 1.0507989919300014, 1.0333524919300012, 1.0469219919300015, 1.0391679919300014, 1.0333524919300012, 1.0352909919300013, 1.0430449919300013]}}

In [ ]:
df = pd.DataFrame(test)

In [ ]:
df